# Test Model Serving via MaaS

This notebook tests model serving through the MaaS inference gateway:
1. Model discovery via MaaS API
2. Inference with API key authentication
3. Streaming support (critical for IDE integration)
4. Authorization enforcement
5. Concurrent requests

> **Rate limiting** is tested separately in `../4_control/2_maas_policy_test.ipynb`.

**Prerequisites:**
- MaaS enabled with model registered (`2_enable_maas.ipynb` completed)
- MaaSModelRef in `Ready` state

In [ ]:
import subprocess, json, time, os
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")

INFERENCE_GW = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")

API_KEY = os.getenv("MAAS_API_KEY", "")
if not API_KEY:
    token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    API_KEY = token_result.stdout.strip()
    print("Using OpenShift token for authentication")
else:
    print(f"Using MaaS API key: {API_KEY[:20]}...")

print(f"\nInference Gateway: {INFERENCE_GW}")
print(f"Model: {MODEL_NAME}")

## 1. Model Discovery

List all models available through the inference gateway.

In [ ]:
import urllib.request, ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    f"{INFERENCE_GW}/v1/models",
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
)

try:
    with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
        models_data = json.loads(resp.read())

    print("Available Models via Inference Gateway")
    print("=" * 60)
    for model in models_data.get("data", []):
        print(f"  {model['id']:<40} {model.get('owned_by', 'N/A')}")

    TEST_MODEL = models_data["data"][0]["id"] if models_data.get("data") else MODEL_NAME
    print(f"\nUsing '{TEST_MODEL}' for tests")
except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.read().decode()[:200]}")
    TEST_MODEL = MODEL_NAME
    print(f"Falling back to: {TEST_MODEL}")
except Exception as e:
    print(f"Error: {e}")
    TEST_MODEL = MODEL_NAME

## 2. Inference Test

Send a chat completion request through the MaaS gateway.

In [ ]:
import httpx
from openai import OpenAI

client = OpenAI(
    base_url=f"{INFERENCE_GW}/v1",
    api_key=API_KEY,
    http_client=httpx.Client(verify=False),
)

print(f"Testing inference on '{TEST_MODEL}' via gateway...")
print("-" * 50)

start = time.time()
response = client.chat.completions.create(
    model=TEST_MODEL,
    messages=[{"role": "user", "content": "Write a Python hello world in one line."}],
    max_tokens=50
)
elapsed = time.time() - start

print(f"Model: {response.model}")
print(f"Response: {response.choices[0].message.content.strip()}")
print(f"Tokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out")
print(f"Latency: {elapsed:.1f}s")

## 3. Streaming Support

Verify streaming works through MaaS (critical for IDE integration — token-by-token delivery).

In [ ]:
print(f"Streaming response from '{TEST_MODEL}':")
print("-" * 50)

start = time.time()
stream = client.chat.completions.create(
    model=TEST_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5, one per line."}],
    max_tokens=50,
    stream=True
)

first_token_time = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if first_token_time is None:
            first_token_time = time.time() - start
        print(chunk.choices[0].delta.content, end="", flush=True)

total_time = time.time() - start
print(f"\n")
print(f"Streaming works")
print(f"  Time to first token: {first_token_time:.2f}s")
print(f"  Total time: {total_time:.2f}s")

## 4. Authorization Enforcement

Verify that unauthenticated requests are rejected (HTTP 401/403).

Ref: [Validation - Test Authorization](https://opendatahub-io.github.io/models-as-a-service/latest/install/validation/)

In [ ]:
print("Authorization Enforcement Tests")
print("=" * 50)

test_url = f"{INFERENCE_GW}/v1/chat/completions"
test_body = json.dumps({"model": TEST_MODEL, "messages": [{"role": "user", "content": "Hi"}], "max_tokens": 5}).encode()

# Test 1: No auth header
print("\n1. No auth header (expecting 401/403):")
req = urllib.request.Request(test_url, data=test_body, headers={"Content-Type": "application/json"}, method="POST")
try:
    with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
        print(f"   WARNING: Got HTTP {resp.status} — auth may NOT be enforced!")
except urllib.error.HTTPError as e:
    if e.code in (401, 403):
        print(f"   PASS — Rejected (HTTP {e.code})")
    else:
        print(f"   Got HTTP {e.code} (expected 401/403)")
except Exception as e:
    print(f"   Error: {e}")

# Test 2: Invalid API key
print("\n2. Invalid API key (expecting 401/403):")
req = urllib.request.Request(test_url, data=test_body,
    headers={"Content-Type": "application/json", "Authorization": "Bearer sk-oai-INVALID-KEY"},
    method="POST")
try:
    with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
        print(f"   WARNING: Got HTTP {resp.status} — invalid key accepted!")
except urllib.error.HTTPError as e:
    if e.code in (401, 403):
        print(f"   PASS — Rejected (HTTP {e.code})")
    else:
        print(f"   Got HTTP {e.code} (expected 401/403)")
except Exception as e:
    print(f"   Error: {e}")

# Test 3: Valid credential
print("\n3. Valid credential (expecting 200):")
req = urllib.request.Request(test_url, data=test_body,
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
    method="POST")
try:
    with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
        print(f"   PASS — Accepted (HTTP {resp.status})")
except urllib.error.HTTPError as e:
    print(f"   Got HTTP {e.code} — check your API key/token")
except Exception as e:
    print(f"   Error: {e}")

## 5. Concurrent Requests

Simulate multiple developers using MaaS simultaneously.

In [ ]:
import concurrent.futures

def make_request(i):
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=TEST_MODEL,
            messages=[{"role": "user", "content": f"Say the number {i}"}],
            max_tokens=5
        )
        elapsed = time.time() - start
        return f"Request {i}: OK ({elapsed:.1f}s)"
    except Exception as e:
        elapsed = time.time() - start
        return f"Request {i}: {str(e)[:50]} ({elapsed:.1f}s)"

print("Sending 5 concurrent requests:")
print("-" * 50)

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(make_request, i) for i in range(5)]
    for future in concurrent.futures.as_completed(futures):
        print(f"  {future.result()}")

## Summary

| Test | What It Validates |
|------|-------------------|
| Model Discovery | MaaS returns available models via `/v1/models` |
| Inference | Chat completion works through MaaS gateway |
| Streaming | Token-by-token delivery for IDE integration |
| Auth Enforcement | Unauthenticated/invalid requests rejected (401/403) |
| Concurrent | Multiple users can access models simultaneously |

## Next Steps

- `4_test_mcp_servers.ipynb` — Test MCP server access through the gateway
- `../4_control/1_maas_advanced.ipynb` — Subscription management, multi-tier demo
- `../4_control/2_maas_policy_test.ipynb` — Rate limit enforcement testing